# recs_015 — D4 cross-encoder rerank on frozen pools

**Follow-up notebook:** [`recs_015_002_ranker_d4_ft_hybrid.ipynb`](recs_015_002_ranker_d4_ft_hybrid.ipynb) — focused spike (FT CE + hybrid tuning). This notebook is the full exploration.

D4 spike on frozen `two_tower_v1` top-100 pools. **Cross-encoder (CE)** scores each `(query_text, candidate_text)` pair jointly (unlike two-tower dot product). Default model: `cross-encoder/ms-marco-MiniLM-L-6-v2`. Candidate text = app name + longest profile review per `app_id`.

## CE variants in this notebook

All D4 methods rerank **within the frozen pool only** (same candidates as `two_tower_v1`). Per-pool **min–max norm** before blending (same as D1). Phase A picks hyperparams on **train_tune** only; val uses fixed weights.

| Method | CE model | Scoring | Tuned on train_tune? |
| --- | --- | --- | --- |
| `two_tower_v1_ranker_d4_cross_encoder_v1` | Zero-shot (MS MARCO) | `norm(ce)` only | No — Phase 0 baseline |
| `two_tower_v1_ranker_d4_ce_retr_blend_v1` | Zero-shot | `w·norm(ce) + (1−w)·norm(retr)` | Yes — grid `w ∈ {0.1,…,1.0}` (CE required) |
| `two_tower_v1_ranker_d4_ce_retr_logpop_blend_v1` | Zero-shot | `w·norm(ce) + (1−w)·(α·norm(retr) + (1−α)·norm(log_pop))` | Yes — grid `w` and `α` |
| `two_tower_v1_ranker_d4_cross_encoder_ft_v1` | **Fine-tuned** on train_fit (Phase B) | `norm(ce)` only | No — weights from BCE training; early-stop on train_tune NDCG |

- **`retr`** = two-tower score from `retrieved_scores_json`
- **`log_pop`** = log1p train popularity within the pool (same prior as D1)
- **`w`** = CE weight in hybrids; **`α`** = retrieval vs popularity inside the `(1−w)` term

**Head-to-head baselines (not CE):** `two_tower_v1` (pool order), `two_tower_v1_heuristic_logpop_blend` (D1), `popularity_train` (full catalog), `two_tower_v1_oracle` (best in pool). Val also reports personalization guardrails (`PersonalizationGapVsPopularity@10`, etc.).

**Promotion bar:** beat tuned **D1** on external val NDCG@10 overall + slice A (no val tuning).

## Phases

- **Phase 0:** precompute zero-shot CE on train_tune + val
- **Phase A:** hybrid blends — tune `w` (and `w`, `α`) on **train_tune** only
- **Phase B:** fine-tune CE on **train_fit**; early-stop on **train_tune** NDCG
- **Val head-to-head:** all D4 variants + baselines + personalization (no val tuning)

**Prereqs:**
```bash
python scripts/recs_job_build_example_cohort.py configs/recs_job_build_example_cohort_train_ranker.json
python scripts/recs_job_export_retrieval_pools.py configs/recs_job_export_retrieval_pools_train_ranker.json
python scripts/recs_job_eval_offline.py configs/recs_job_eval_offline.json \
  --examples-parquet artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet
pip install -e '.[cross-encoder]'
```

See [`docs/ranker_exploration_plan.md`](../../docs/ranker_exploration_plan.md) § D4. Compare with [`recs_014_ranker_d2_d3_train_head_to_head.ipynb`](recs_014_ranker_d2_d3_train_head_to_head.ipynb).

## Cost note

Cross-encoding is **~100 forward passes per example**. CE scores are **precomputed once** per model variant (zero-shot + fine-tuned) and reused for tuning and val — avoid re-running the model inside the per-method val loop.

Defaults to **full val** (12.5k). For smoke only, set `MAX_VAL_EXAMPLES = 500` (not comparable to official metrics). Phase B fine-tuning is slow; set `TRAIN_CE_FT = False` to skip.

In [1]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from steam_review_ml.evaluation.example_cohort import (
    cohort_parquet_path,
    load_retrieval_pool_rows,
    load_retrieval_pools_jsonl,
)
from steam_review_ml.evaluation.heuristic_ranker import (
    METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND,
    pool_rerank_registry,
    rerank_scores_on_pool,
)
from steam_review_ml.evaluation.retrieval_offline_eval import (
    _oracle_ranked_indices_from_retrieved,
    _rank_rows,
    _table_personalization,
    average_precision_at_k,
    hit_rate_at_k,
    load_eval_examples_from_parquet,
    load_ranking_catalog_context,
    mrr,
    ndcg_at_k,
)
from steam_review_ml.recommender.retrieve import ContentRetriever
from steam_review_ml.recommender.ranker_d4_cross_encoder import (
    METHOD_TWO_TOWER_V1_RANKER_D4_CE_RETR_BLEND_V1,
    METHOD_TWO_TOWER_V1_RANKER_D4_CE_RETR_LOGPOP_BLEND_V1,
    METHOD_TWO_TOWER_V1_RANKER_D4_CROSS_ENCODER_FT_V1,
    METHOD_TWO_TOWER_V1_RANKER_D4_CROSS_ENCODER_V1,
    CrossEncoderConfig,
    CrossEncoderFinetuneConfig,
    build_app_candidate_texts,
    load_cross_encoder,
    make_ce_cache_score_fn,
    make_ce_retr_blend_score_fn,
    make_ce_retr_logpop_blend_score_fn,
    precompute_ce_scores_by_ex_idx,
    query_text_lookup_from_cohort_parquet,
    query_text_map_from_examples,
    query_text_map_from_train_pools,
    train_cross_encoder_ranker,
    tune_ce_retr_blend,
    tune_ce_retr_logpop_blend,
)

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

POOL_METHOD = "two_tower_v1"
K_FINAL = 10
K_PERSONALIZATION = 10
MIN_REVIEW_CHARS = 30
ARTIFACT_DIR = REPO_ROOT / "artifacts/recs"
SPLIT_SEED = 2027
TUNE_FRAC = 0.10
MAX_VAL_EXAMPLES: int | None = None
TRAIN_CE_FT = True

TRAIN_POOLS_PARQUET = REPO_ROOT / "artifacts/recs/ranker_pools/train_ranker_v1/two_tower_v1.parquet"
TRAIN_COHORT_PARQUET = cohort_parquet_path(REPO_ROOT / "artifacts/recs/eval_cache/train_ranker_v1")
VAL_JSONL = REPO_ROOT / "artifacts/recs/offline_eval/runs/latest/eval_offline_examples.jsonl"
VAL_COHORT_PARQUET = cohort_parquet_path(REPO_ROOT / "artifacts/recs/eval_cache/val_dev_12k_v1")
GAME_PROFILE_REVIEWS = REPO_ROOT / "artifacts/recs/embeddings/game_profile/default/game_profile_reviews.parquet"
D4_FT_OUTPUT_DIR = REPO_ROOT / "artifacts/recs/rankers/d4_cross_encoder_ft_v1"

CE_CFG = CrossEncoderConfig()
CE_FT_CFG = CrossEncoderFinetuneConfig(epochs=7, early_stopping_patience=2)

for p in (TRAIN_POOLS_PARQUET, TRAIN_COHORT_PARQUET, VAL_JSONL, VAL_COHORT_PARQUET, GAME_PROFILE_REVIEWS):
    if not p.is_file():
        raise FileNotFoundError(f"Missing required artifact: {p}")

print(f"TRAIN_POOLS={TRAIN_POOLS_PARQUET}")
print(f"VAL_JSONL={VAL_JSONL}")
print(f"CE model={CE_CFG.model_name}")
print(f"MAX_VAL_EXAMPLES={MAX_VAL_EXAMPLES}  TRAIN_CE_FT={TRAIN_CE_FT}")

TRAIN_POOLS=/home/ryanr/workspace/steam_recommendations/artifacts/recs/ranker_pools/train_ranker_v1/two_tower_v1.parquet
VAL_JSONL=/home/ryanr/workspace/steam_recommendations/artifacts/recs/offline_eval/runs/latest/eval_offline_examples.jsonl
CE model=cross-encoder/ms-marco-MiniLM-L-6-v2
MAX_VAL_EXAMPLES=None  TRAIN_CE_FT=True


/home/ryanr/miniconda3/envs/tf_condaforge/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Load pools + catalog + query text

In [2]:
train_pools = load_retrieval_pool_rows(TRAIN_POOLS_PARQUET)
val_pools = load_retrieval_pools_jsonl(VAL_JSONL, method=POOL_METHOD)
if MAX_VAL_EXAMPLES is not None:
    val_pools = val_pools[: int(MAX_VAL_EXAMPLES)]

val_examples = load_eval_examples_from_parquet(VAL_COHORT_PARQUET)
train_query_lookup = query_text_lookup_from_cohort_parquet(TRAIN_COHORT_PARQUET)
train_query_by_ex_idx = query_text_map_from_train_pools(train_pools, examples_by_key=train_query_lookup)
val_query_by_ex_idx = query_text_map_from_examples(val_examples)

candidate_texts = build_app_candidate_texts(
    GAME_PROFILE_REVIEWS,
    max_chars_per_app=CE_CFG.candidate_text_max_chars,
)

catalog = load_ranking_catalog_context(
    repo_root=REPO_ROOT,
    min_review_chars=MIN_REVIEW_CHARS,
    artifact_dir=ARTIFACT_DIR,
)
app_ids = catalog.app_ids
app_to_row = catalog.app_to_row
pop_row = catalog.pop_row

print(
    f"train pools: {len(train_pools):,}  val pools: {len(val_pools):,}  "
    f"catalog apps: {len(app_ids):,}  candidate docs: {len(candidate_texts):,}"
)

train pools: 51,691  val pools: 12,500  catalog apps: 315  candidate docs: 315


## Train fit / tune split (90/10, stratified by slice)

Same discipline as `recs_014`: tune hybrids and CE fine-tune early-stop on **train_tune** only. External val is report-only.

In [3]:
def stratified_ex_idx_split(
    pools: list[dict[str, Any]], *, tune_frac: float, seed: int
) -> tuple[set[int], set[int]]:
    rng = np.random.default_rng(seed)
    by_slice: dict[str, list[int]] = {}
    for row in pools:
        by_slice.setdefault(str(row["slice_name"]), []).append(int(row["ex_idx"]))
    fit_ids: set[int] = set()
    tune_ids: set[int] = set()
    for _slice, ids in by_slice.items():
        ids_arr = np.asarray(sorted(ids))
        rng.shuffle(ids_arr)
        n_tune = max(1, int(round(len(ids_arr) * tune_frac)))
        tune_ids.update(int(x) for x in ids_arr[:n_tune])
        fit_ids.update(int(x) for x in ids_arr[n_tune:])
    return fit_ids, tune_ids


fit_ex_idx, tune_ex_idx = stratified_ex_idx_split(train_pools, tune_frac=TUNE_FRAC, seed=SPLIT_SEED)
train_fit = [r for r in train_pools if int(r["ex_idx"]) in fit_ex_idx]
train_tune = [r for r in train_pools if int(r["ex_idx"]) in tune_ex_idx]
print(f"fit={len(train_fit):,}  tune={len(train_tune):,}")

fit=46,522  tune=5,169


## Shared helpers (pool → top-K metrics)

In [4]:
def pool_scores_to_ranked_indices(
    pool_app_ids: list[int],
    pool_scores: np.ndarray,
    *,
    k_final: int,
) -> np.ndarray:
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, score in zip(pool_app_ids, pool_scores):
        full[int(app_to_row[int(app_id)])] = float(score)
    return _rank_rows(full)[:k_final]


def popularity_catalog_scores(*, query_app_id: int) -> np.ndarray:
    s = np.asarray(pop_row, dtype=np.float64).copy()
    row = app_to_row.get(int(query_app_id))
    if row is not None:
        s[row] = -np.inf
    return s


def popularity_catalog_ranked_indices(*, query_app_id: int, k_final: int) -> np.ndarray:
    return _rank_rows(popularity_catalog_scores(query_app_id=query_app_id))[:k_final]


def full_catalog_scores_for_pool_row(
    row: dict[str, Any],
    *,
    score_fn: Callable[..., np.ndarray] | None = None,
    catalog_pop: bool = False,
) -> np.ndarray:
    if catalog_pop:
        return popularity_catalog_scores(query_app_id=int(row["query_app_id"]))
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    if score_fn is None:
        pool_scores = np.asarray(ret_sc, dtype=np.float64)
    else:
        pool_scores = np.asarray(
            score_fn(pool_apps, ret_sc, ex_idx=int(row["ex_idx"])), dtype=np.float64
        )
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, score in zip(pool_apps, pool_scores):
        full[int(app_to_row[int(app_id)])] = float(score)
    return full


def per_example_metrics(
    row: dict[str, Any],
    *,
    method: str,
    score_fn: Callable[..., np.ndarray] | None = None,
    oracle: bool = False,
    catalog_pop: bool = False,
) -> dict[str, Any] | None:
    positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
    if not positives:
        return None

    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    retrieved_rows = np.asarray([app_to_row[a] for a in pool_apps], dtype=np.int64)

    if oracle:
        ranked = _oracle_ranked_indices_from_retrieved(retrieved_rows, positives, app_ids)[:K_FINAL]
    elif catalog_pop:
        ranked = popularity_catalog_ranked_indices(query_app_id=int(row["query_app_id"]), k_final=K_FINAL)
    elif score_fn is None:
        ranked = pool_scores_to_ranked_indices(pool_apps, np.asarray(ret_sc), k_final=K_FINAL)
    else:
        blend = score_fn(pool_apps, ret_sc, ex_idx=int(row["ex_idx"]))
        ranked = pool_scores_to_ranked_indices(pool_apps, blend, k_final=K_FINAL)

    return {
        "method": method,
        "slice_name": row.get("slice_name", ""),
        "Hit@K": hit_rate_at_k(ranked, positives, K_FINAL, app_ids),
        "MAP@K": average_precision_at_k(ranked, positives, K_FINAL, app_ids),
        "NDCG@K": ndcg_at_k(ranked, positives, K_FINAL, app_ids),
        "MRR": mrr(ranked, positives, app_ids),
    }

## Phase 0 — Precompute zero-shot CE scores

Run once on **train_tune** (for hybrid tuning) and **val** (for reporting).

In [5]:
ce_model_zs = load_cross_encoder(CE_CFG.model_name)

print("Precomputing zero-shot CE on train_tune...")
ce_tune_zs = precompute_ce_scores_by_ex_idx(
    ce_model_zs,
    train_tune,
    query_text_by_ex_idx=train_query_by_ex_idx,
    candidate_texts=candidate_texts,
    cfg=CE_CFG,
)

print("Precomputing zero-shot CE on val...")
ce_val_zs = precompute_ce_scores_by_ex_idx(
    ce_model_zs,
    val_pools,
    query_text_by_ex_idx=val_query_by_ex_idx,
    candidate_texts=candidate_texts,
    cfg=CE_CFG,
)
print(f"cached tune={len(ce_tune_zs):,}  val={len(ce_val_zs):,} pools")

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 3100.13it/s]


Precomputing zero-shot CE on train_tune...
  CE precompute 50/5,169...
  CE precompute 100/5,169...
  CE precompute 150/5,169...
  CE precompute 200/5,169...
  CE precompute 250/5,169...
  CE precompute 300/5,169...
  CE precompute 350/5,169...
  CE precompute 400/5,169...
  CE precompute 450/5,169...
  CE precompute 500/5,169...
  CE precompute 550/5,169...
  CE precompute 600/5,169...
  CE precompute 650/5,169...
  CE precompute 700/5,169...
  CE precompute 750/5,169...
  CE precompute 800/5,169...
  CE precompute 850/5,169...
  CE precompute 900/5,169...
  CE precompute 950/5,169...
  CE precompute 1,000/5,169...
  CE precompute 1,050/5,169...
  CE precompute 1,100/5,169...
  CE precompute 1,150/5,169...
  CE precompute 1,200/5,169...
  CE precompute 1,250/5,169...
  CE precompute 1,300/5,169...
  CE precompute 1,350/5,169...
  CE precompute 1,400/5,169...
  CE precompute 1,450/5,169...
  CE precompute 1,500/5,169...
  CE precompute 1,550/5,169...
  CE precompute 1,600/5,169...
  CE

## Phase A — Hybrid rerank (tune on train_tune)

Grid `w ∈ {0.1, 0.2, …, 2.0}` for CE+retr (CE weight must be &gt; 0); same `w` grid plus `α` for CE+retr+logpop. Fixed params applied to val below.

In [6]:
best_w, tune_ndcg_ce_retr = tune_ce_retr_blend(
    train_tune,
    ce_tune_zs,
    app_ids=app_ids,
    app_to_row=app_to_row,
    k_final=K_FINAL,
)
print(f"ce_retr_blend: best_w={best_w:.1f}  train_tune NDCG@{K_FINAL}={tune_ndcg_ce_retr:.4f}")

best_w_lp, best_alpha, tune_ndcg_ce_retr_lp = tune_ce_retr_logpop_blend(
    train_tune,
    ce_tune_zs,
    pop_row=pop_row,
    app_to_row=app_to_row,
    app_ids=app_ids,
    k_final=K_FINAL,
)
print(
    f"ce_retr_logpop_blend: best_w={best_w_lp:.1f}  best_alpha={best_alpha:.1f}  "
    f"train_tune NDCG@{K_FINAL}={tune_ndcg_ce_retr_lp:.4f}"
)

ce_retr_blend: best_w=1.2  train_tune NDCG@10=0.0515
ce_retr_logpop_blend: best_w=0.1  best_alpha=0.1  train_tune NDCG@10=0.1640


## Phase B — Fine-tune cross-encoder (optional)

Pairwise BCE on in-pool negatives from **train_fit**; early-stop on **train_tune** CE-only NDCG. Skips if `TRAIN_CE_FT=False` or a saved model exists at `D4_FT_OUTPUT_DIR/cross_encoder_model`.

In [7]:
ce_val_ft: dict[int, np.ndarray] | None = None
ft_history: pd.DataFrame | None = None
saved_ft = D4_FT_OUTPUT_DIR / "cross_encoder_model"

if TRAIN_CE_FT:
    if saved_ft.is_dir():
        print(f"Loading saved fine-tuned CE from {saved_ft}")
        ce_model_ft = load_cross_encoder(model_path=saved_ft)
        # to delete:
        # rm -rf artifacts/recs/rankers/d4_cross_encoder_ft_v1/cross_encoder_model
        # rm -rf artifacts/recs/rankers/d4_cross_encoder_ft_v1/_best_ce_checkpoint
    else:
        print("Fine-tuning cross-encoder on train_fit...")
        ce_model_ft, ft_history = train_cross_encoder_ranker(
            fit_pools=train_fit,
            tune_pools=train_tune,
            query_text_by_ex_idx=train_query_by_ex_idx,
            candidate_texts=candidate_texts,
            app_ids=app_ids,
            app_to_row=app_to_row,
            output_dir=D4_FT_OUTPUT_DIR,
            cfg=CE_FT_CFG,
            score_cfg=CE_CFG,
            k_final=K_FINAL,
        )
        display(ft_history)

    print("Precomputing fine-tuned CE on val...")
    ce_val_ft = precompute_ce_scores_by_ex_idx(
        ce_model_ft,
        val_pools,
        query_text_by_ex_idx=val_query_by_ex_idx,
        candidate_texts=candidate_texts,
        cfg=CE_CFG,
    )
else:
    print("TRAIN_CE_FT=False — skipping Phase B")

Loading saved fine-tuned CE from /home/ryanr/workspace/steam_recommendations/artifacts/recs/rankers/d4_cross_encoder_ft_v1/cross_encoder_model


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2430.74it/s]


Precomputing fine-tuned CE on val...
  CE precompute 50/12,500...
  CE precompute 100/12,500...
  CE precompute 150/12,500...
  CE precompute 200/12,500...
  CE precompute 250/12,500...
  CE precompute 300/12,500...
  CE precompute 350/12,500...
  CE precompute 400/12,500...
  CE precompute 450/12,500...
  CE precompute 500/12,500...
  CE precompute 550/12,500...
  CE precompute 600/12,500...
  CE precompute 650/12,500...
  CE precompute 700/12,500...
  CE precompute 750/12,500...
  CE precompute 800/12,500...
  CE precompute 850/12,500...
  CE precompute 900/12,500...
  CE precompute 950/12,500...
  CE precompute 1,000/12,500...
  CE precompute 1,050/12,500...
  CE precompute 1,100/12,500...
  CE precompute 1,150/12,500...
  CE precompute 1,200/12,500...
  CE precompute 1,250/12,500...
  CE precompute 1,300/12,500...
  CE precompute 1,350/12,500...
  CE precompute 1,400/12,500...
  CE precompute 1,450/12,500...
  CE precompute 1,500/12,500...
  CE precompute 1,550/12,500...
  CE preco

## Val head-to-head (cached CE scores)

All D4 variants use precomputed score vectors — no per-method CE forward passes.

In [8]:
D1_SPEC = pool_rerank_registry()[METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND]


def score_d1_logpop(pool_apps, retrieval_scores, **_ignored):
    return rerank_scores_on_pool(
        pool_apps,
        retrieval_scores,
        D1_SPEC,
        pop_row=pop_row,
        app_to_row=app_to_row,
    )


HEAD_TO_HEAD: list[dict[str, Any]] = [
    {"method": POOL_METHOD, "kind": "pool_retrieval"},
    {"method": METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND, "kind": "pool_rerank", "score_fn": score_d1_logpop},
    {
        "method": METHOD_TWO_TOWER_V1_RANKER_D4_CROSS_ENCODER_V1,
        "kind": "pool_rerank",
        "score_fn": make_ce_cache_score_fn(ce_val_zs),
    },
    {
        "method": METHOD_TWO_TOWER_V1_RANKER_D4_CE_RETR_BLEND_V1,
        "kind": "pool_rerank",
        "score_fn": make_ce_retr_blend_score_fn(ce_val_zs, w=best_w),
    },
    {
        "method": METHOD_TWO_TOWER_V1_RANKER_D4_CE_RETR_LOGPOP_BLEND_V1,
        "kind": "pool_rerank",
        "score_fn": make_ce_retr_logpop_blend_score_fn(
            ce_val_zs,
            pop_row=pop_row,
            app_to_row=app_to_row,
            w=best_w_lp,
            alpha=best_alpha,
        ),
    },
    {"method": "popularity_train", "kind": "catalog_pop"},
    {"method": f"{POOL_METHOD}_oracle", "kind": "oracle"},
]

if ce_val_ft is not None:
    HEAD_TO_HEAD.insert(
        -2,
        {
            "method": METHOD_TWO_TOWER_V1_RANKER_D4_CROSS_ENCODER_FT_V1,
            "kind": "pool_rerank",
            "score_fn": make_ce_cache_score_fn(ce_val_ft),
        },
    )

val_rows: list[dict[str, Any]] = []
for i, row in enumerate(val_pools):
    if i and i % 500 == 0:
        print(f"scored {i:,}/{len(val_pools):,} examples...", flush=True)
    for spec in HEAD_TO_HEAD:
        kind = spec["kind"]
        if kind == "oracle":
            m = per_example_metrics(row, method=spec["method"], oracle=True)
        elif kind == "catalog_pop":
            m = per_example_metrics(row, method=spec["method"], catalog_pop=True)
        elif kind == "pool_retrieval":
            m = per_example_metrics(row, method=spec["method"], score_fn=None)
        elif kind == "pool_rerank":
            m = per_example_metrics(row, method=spec["method"], score_fn=spec["score_fn"])
        else:
            continue
        if m is not None:
            val_rows.append(m)

df_val = pd.DataFrame(val_rows)
print(f"methods: {sorted(df_val['method'].unique())}")

scored 500/12,500 examples...
scored 1,000/12,500 examples...
scored 1,500/12,500 examples...
scored 2,000/12,500 examples...
scored 2,500/12,500 examples...
scored 3,000/12,500 examples...
scored 3,500/12,500 examples...
scored 4,000/12,500 examples...
scored 4,500/12,500 examples...
scored 5,000/12,500 examples...
scored 5,500/12,500 examples...
scored 6,000/12,500 examples...
scored 6,500/12,500 examples...
scored 7,000/12,500 examples...
scored 7,500/12,500 examples...
scored 8,000/12,500 examples...
scored 8,500/12,500 examples...
scored 9,000/12,500 examples...
scored 9,500/12,500 examples...
scored 10,000/12,500 examples...
scored 10,500/12,500 examples...
scored 11,000/12,500 examples...
scored 11,500/12,500 examples...
scored 12,000/12,500 examples...
methods: ['popularity_train', 'two_tower_v1', 'two_tower_v1_heuristic_logpop_blend', 'two_tower_v1_oracle', 'two_tower_v1_ranker_d4_ce_retr_blend_v1', 'two_tower_v1_ranker_d4_ce_retr_logpop_blend_v1', 'two_tower_v1_ranker_d4_cros

## Val personalization (same contract as recs_011 / eval job)

In [9]:
for i, ex in enumerate(val_examples):
    ex["ex_idx"] = i

pools_by_ex = {int(r["ex_idx"]): r for r in val_pools}
pool_ex_indices = set(pools_by_ex.keys())


def _make_pool_catalog_scorer(
    score_fn: Callable[..., np.ndarray] | None = None,
) -> Callable[[dict[str, Any]], np.ndarray]:
    def scorer(ex: dict[str, Any]) -> np.ndarray:
        return full_catalog_scores_for_pool_row(
            pools_by_ex[int(ex["ex_idx"])], score_fn=score_fn
        )

    return scorer


person_methods: dict[str, Callable[[dict[str, Any]], np.ndarray]] = {
    "popularity_train": lambda ex: popularity_catalog_scores(
        query_app_id=int(ex["query_app_id"])
    ),
}

for spec in HEAD_TO_HEAD:
    if spec["kind"] == "oracle":
        continue
    if spec["kind"] == "catalog_pop":
        continue
    if spec["kind"] == "pool_retrieval":
        person_methods[spec["method"]] = _make_pool_catalog_scorer(score_fn=None)
    elif spec["kind"] == "pool_rerank":
        person_methods[spec["method"]] = _make_pool_catalog_scorer(score_fn=spec["score_fn"])

retriever = ContentRetriever(artifact_dir=ARTIFACT_DIR, repo_root=REPO_ROOT)
df_person = _table_personalization(
    methods=person_methods,
    examples=val_examples,
    X=retriever.embedding_matrix,
    app_ids=app_ids,
    pop_row=pop_row,
    k_personalization=K_PERSONALIZATION,
    example_indices=pool_ex_indices,
)

person_cols = [
    f"ILD@{K_PERSONALIZATION}",
    f"CatalogCoverage@{K_PERSONALIZATION}",
    f"Novelty@{K_PERSONALIZATION}",
    f"PersonalizationGapVsPopularity@{K_PERSONALIZATION}",
]
df_val_overall = (
    df_val.groupby("method")[["Hit@K", "MAP@K", "NDCG@K", "MRR"]]
    .mean()
    .reset_index()
)
df_val_full = df_val_overall.merge(df_person, on="method", how="left")
gap_col = f"PersonalizationGapVsPopularity@{K_PERSONALIZATION}"
print(f"personalization methods: {len(df_person)}")

personalization methods: 7


In [10]:
display(Markdown("### Val relevance + personalization (overall)"))
display(
    df_val_full.sort_values("NDCG@K", ascending=False)[
        ["method", "Hit@K", "NDCG@K", "MRR", *person_cols]
    ]
)

display(Markdown("### Val ranking by slice"))
display(
    df_val.groupby(["slice_name", "method"])[["Hit@K", "NDCG@K", "MRR"]]
    .mean()
    .sort_values(["slice_name", "NDCG@K"], ascending=[True, False])
)

display(Markdown("### Personalization only (sorted by gap vs popularity)"))
display(df_person.sort_values(gap_col, ascending=False))

d1_ndcg = float(
    df_val.loc[df_val["method"] == METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND, "NDCG@K"].mean()
)
d4_methods = [
    METHOD_TWO_TOWER_V1_RANKER_D4_CROSS_ENCODER_V1,
    METHOD_TWO_TOWER_V1_RANKER_D4_CE_RETR_BLEND_V1,
    METHOD_TWO_TOWER_V1_RANKER_D4_CE_RETR_LOGPOP_BLEND_V1,
    METHOD_TWO_TOWER_V1_RANKER_D4_CROSS_ENCODER_FT_V1,
]
print(f"\nD1 logpop val NDCG@{K_FINAL}={d1_ndcg:.4f}")
for m in d4_methods:
    if m not in df_val["method"].values:
        continue
    nd = float(df_val.loc[df_val["method"] == m, "NDCG@K"].mean())
    beat = "BEATS D1" if nd > d1_ndcg else "below D1"
    print(f"  {m}: NDCG@{K_FINAL}={nd:.4f} ({beat})")

display(Markdown("### D4 vs D1 — relevance vs personalization"))
d1_row = df_val_full.loc[
    df_val_full["method"] == METHOD_TWO_TOWER_V1_HEURISTIC_LOGPOP_BLEND
].iloc[0]
for m in [
    METHOD_TWO_TOWER_V1_RANKER_D4_CE_RETR_LOGPOP_BLEND_V1,
    METHOD_TWO_TOWER_V1_RANKER_D4_CROSS_ENCODER_FT_V1,
    METHOD_TWO_TOWER_V1_RANKER_D4_CROSS_ENCODER_V1,
    POOL_METHOD,
]:
    if m not in df_val_full["method"].values:
        continue
    r = df_val_full.loc[df_val_full["method"] == m].iloc[0]
    ndcg_delta = float(r["NDCG@K"]) - float(d1_row["NDCG@K"])
    gap_delta = float(r[gap_col]) - float(d1_row[gap_col])
    print(f"  {m}: ΔNDCG vs D1={ndcg_delta:+.4f}  Δgap vs D1={gap_delta:+.4f}")

### Val relevance + personalization (overall)

,method,Hit@K,NDCG@K,MRR,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
3,two_tower_v1_oracle,0.51224,0.498310,0.512240,NaN,NaN,NaN,NaN
2,two_tower_v1_heuristic_logpop_blend,0.19328,0.092892,0.067059,0.204632,0.501587,6.272161,0.720097
5,two_tower_v1_ranker_d4_ce_retr_logpop_blend_v1,0.19208,0.090581,0.064617,0.207834,0.495238,6.235568,0.713886
0,popularity_train,0.15112,0.073109,0.052221,0.212335,0.034921,5.317471,0.000000
6,two_tower_v1_ranker_d4_cross_encoder_ft_v1,0.14992,0.066807,0.045846,0.206847,0.400000,6.507556,0.792981
7,two_tower_v1_ranker_d4_cross_encoder_v1,0.07576,0.036281,0.027276,0.222953,1.000000,9.574959,0.965979
4,two_tower_v1_ranker_d4_ce_retr_blend_v1,0.07520,0.035710,0.026613,0.221392,1.000000,9.427321,0.964971
1,two_tower_v1,0.04680,0.018161,0.011008,0.261639,0.987302,12.167556,0.995623


### Val ranking by slice

Hit@K  \
slice_name            method                                                     
slice_a_multi_target  two_tower_v1_oracle                             0.773793   
                      two_tower_v1_ranker_d4_ce_retr_logpop_blend_v1  0.291034   
                      two_tower_v1_heuristic_logpop_blend             0.271724   
                      two_tower_v1_ranker_d4_cross_encoder_ft_v1      0.228966   
                      two_tower_v1_ranker_d4_cross_encoder_v1         0.158621   
                      two_tower_v1_ranker_d4_ce_retr_blend_v1         0.151724   
                      popularity_train                                0.128276   
                      two_tower_v1                                    0.091034   
slice_b_single_target two_tower_v1_oracle                             0.496136   
                      two_tower_v1_heuristic_logpop_blend             0.188450   
                      two_tower_v1_ranker_d4_ce_retr_logpop_blend_v1  0.185987   
                      popularity_train                                0.152527   
                      two_tower_v1_ranker_d4_cross_encoder_ft_v1      0.145053   
                      two_tower_v1_ranker_d4_cross_encoder_v1         0.070658   
                      two_tower_v1_ranker_d4_ce_retr_blend_v1         0.070488   
                      two_tower_v1                                    0.044076   

                                                                        NDCG@K  \
slice_name            method                                                     
slice_a_multi_target  two_tower_v1_oracle                             0.533618   
                      two_tower_v1_ranker_d4_ce_retr_logpop_blend_v1  0.070331   
                      two_tower_v1_heuristic_logpop_blend             0.068322   
                      two_tower_v1_ranker_d4_cross_encoder_ft_v1      0.057766   
                      two_tower_v1_ranker_d4_cross_encoder_v1         0.038853   
                      two_tower_v1_ranker_d4_ce_retr_blend_v1         0.038250   
                      popularity_train                                0.035444   
                      two_tower_v1                                    0.020537   
slice_b_single_target two_tower_v1_oracle                             0.496136   
                      two_tower_v1_heuristic_logpop_blend             0.094404   
                      two_tower_v1_ranker_d4_ce_retr_logpop_blend_v1  0.091827   
                      popularity_train                                0.075428   
                      two_tower_v1_ranker_d4_cross_encoder_ft_v1      0.067364   
                      two_tower_v1_ranker_d4_cross_encoder_v1         0.036122   
                      two_tower_v1_ranker_d4_ce_retr_blend_v1         0.035553   
                      two_tower_v1                                    0.018015   

                                                                           MRR  
slice_name            method                                                    
slice_a_multi_target  two_tower_v1_oracle                             0.773793  
                      two_tower_v1_ranker_d4_ce_retr_logpop_blend_v1  0.081905  
                      two_tower_v1_heuristic_logpop_blend             0.081396  
                      two_tower_v1_ranker_d4_cross_encoder_ft_v1      0.072900  
                      two_tower_v1_ranker_d4_cross_encoder_v1         0.051230  
                      two_tower_v1_ranker_d4_ce_retr_blend_v1         0.051025  
                      popularity_train                                0.047469  
                      two_tower_v1                                    0.021192  
slice_b_single_target two_tower_v1_oracle                             0.496136  
                      two_tower_v1_heuristic_logpop_blend             0.066176  
                      two_tower_v1_ranker_d4_ce_retr_logpop_blend_v1  0.063553  
                      popularity_train                            

### Personalization only (sorted by gap vs popularity)

,method,ILD@10,CatalogCoverage@10,Novelty@10,PersonalizationGapVsPopularity@10
1,two_tower_v1,0.261639,0.987302,12.167556,0.995623
6,two_tower_v1_ranker_d4_cross_encoder_v1,0.222953,1.000000,9.574959,0.965979
3,two_tower_v1_ranker_d4_ce_retr_blend_v1,0.221392,1.000000,9.427321,0.964971
5,two_tower_v1_ranker_d4_cross_encoder_ft_v1,0.206847,0.400000,6.507556,0.792981
2,two_tower_v1_heuristic_logpop_blend,0.204632,0.501587,6.272161,0.720097
4,two_tower_v1_ranker_d4_ce_retr_logpop_blend_v1,0.207834,0.495238,6.235568,0.713886
0,popularity_train,0.212335,0.034921,5.317471,0.000000



D1 logpop val NDCG@10=0.0929
  two_tower_v1_ranker_d4_cross_encoder_v1: NDCG@10=0.0363 (below D1)
  two_tower_v1_ranker_d4_ce_retr_blend_v1: NDCG@10=0.0357 (below D1)
  two_tower_v1_ranker_d4_ce_retr_logpop_blend_v1: NDCG@10=0.0906 (below D1)
  two_tower_v1_ranker_d4_cross_encoder_ft_v1: NDCG@10=0.0668 (below D1)


### D4 vs D1 — relevance vs personalization

  two_tower_v1_ranker_d4_ce_retr_logpop_blend_v1: ΔNDCG vs D1=-0.0023  Δgap vs D1=-0.0062
  two_tower_v1_ranker_d4_cross_encoder_ft_v1: ΔNDCG vs D1=-0.0261  Δgap vs D1=+0.0729
  two_tower_v1_ranker_d4_cross_encoder_v1: ΔNDCG vs D1=-0.0566  Δgap vs D1=+0.2459
  two_tower_v1: ΔNDCG vs D1=-0.0747  Δgap vs D1=+0.2755


### Takeaway

- **Zero-shot CE** (MS MARCO MiniLM) is a domain-mismatch probe — expect it to lose to D1 unless hybrid blends recover retr/pop signal.
- **Phase A hybrids** tune `w` / `α` on train_tune only; check whether blending CE with retrieval (and logpop) closes the gap on full val.
- **Phase B fine-tune** adapts CE to Steam query/doc pairs; promotion bar is beating **tuned D1** on external val NDCG@10 overall + slice A before wiring into the eval job.
- **Personalization** (`PersonalizationGapVsPopularity@10`, `Novelty@10`, `ILD@10`, `CatalogCoverage@10`): compare relevance vs anti-popularity tradeoff. If `ce_retr_logpop` is close to D1 on NDCG but higher gap, consider a finer `w`/`α` grid spike; otherwise kill D4.
- If no D4 variant wins on relevance or sits on a better frontier: bi-encoder + shallow ranker (D1/D2) remains the cost/quality sweet spot for v1.